# Custom Hook Composition

This 50-minute-track notebook is intentionally self-contained: it uses a seeded tiny PyTorch model, no downloads, and only public TorchLens APIs. The goal is to show the full workflow shape you would use on a larger model while keeping every cell runnable on CPU.


Built-in helpers cover common interventions, but custom hooks are the escape hatch for domain-specific edits. A hook receives the activation and a TorchLens hook context; it must return a tensor with a compatible shape.


In [ ]:
from __future__ import annotations

import math
from typing import Any

import matplotlib.pyplot as plt
import torch
from torch import nn

import torchlens as tl

SEED = 1609
torch.manual_seed(SEED)
torch.set_grad_enabled(False)


class TinyHookModel(nn.Module):
    """Small model for hook composition."""

    def __init__(self) -> None:
        """Initialize layers."""
        super().__init__()
        self.proj = nn.Linear(5, 5)
        self.out = nn.Linear(5, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run one hidden ReLU before the readout."""
        return self.out(torch.relu(self.proj(x)))


def center_activation(activation: torch.Tensor, *, hook: Any) -> torch.Tensor:
    """Subtract the feature mean from an activation."""
    return activation - activation.mean(dim=-1, keepdim=True)


def cap_positive(activation: torch.Tensor, *, hook: Any) -> torch.Tensor:
    """Clamp positive activations to a small ceiling."""
    return activation.clamp(max=0.25)

In [ ]:
model = TinyHookModel().eval()
x = torch.randn(3, 5)
clean = tl.log_forward_pass(model, x, vis_opt="none", intervention_ready=True, name="clean")
site = clean.find_sites(tl.func("relu")).first()
print(site.layer_label)

In [ ]:
composed = clean.fork("center_then_cap")
composed.attach_hooks(tl.label(site.layer_label), center_activation, cap_positive)
composed.replay()

clean_hidden = clean[site.layer_label].activation
composed_hidden = composed[site.layer_label].activation
print(f"clean hidden mean: {float(clean_hidden.mean()): .4f}")
print(f"composed hidden mean: {float(composed_hidden.mean()): .4f}")
print(f"composed hidden max: {float(composed_hidden.max()): .4f}")

In [ ]:
with clean.fork("temporary_noise") as scoped_log:
    scoped_log.attach_hooks(tl.label(site.layer_label), tl.noise(std=0.01)).replay()
    noisy_delta = torch.linalg.vector_norm(
        scoped_log.layer_list[-1].activation - clean.layer_list[-1].activation
    )
print(f"temporary hook output delta: {noisy_delta:.4f}")

Composition order is left to right in `attach_hooks`. Keep custom hooks small, deterministic, and shape-preserving unless the downstream graph explicitly supports a shape change.
